# Subsetting and Filtering

In this tutorial, you’ll learn how to select subsets of an `xsnowDataset` along its dimensions by using standard `xarray` indexing.

After this tutorial, you will be able to:

1) Select data by **location**, **time**, **slope**, **realization**, or **layer**
2) Mask and filter data by **variable conditions**
3) Explore the `xsnow` classification extension for solving domain-specific, yet generic tasks

We use a small sample SNOWPACK dataset shipped with `xsnow.sample_data`.

In [1]:
import xsnow
ds = xsnow.sample_data.snp_gridded_ds()
print(ds)

<xsnowDataset>
  Locations:  5
  Timestamps: 416 (2024-01-16--2024-02-02)
  Profiles:   NotImplementedYet

    employing the <xarray.Dataset> Size: 42MB
    Dimensions:                   (location: 5, time: 416, slope: 5,
                                   realization: 1, layer: 33)
    Coordinates:
      * location                  (location) int64 40B 0 1 2 3 4
      * time                      (time) datetime64[ns] 3kB 2024-01-16T05:00:00 ....
      * realization               (realization) int64 8B 0
      * layer                     (layer) int64 264B 0 1 2 3 4 5 ... 28 29 30 31 32
      * slope                     (slope) int64 40B 0 1 2 3 4
        latitude                  (location) float64 40B 47.15 47.44 ... 47.44 47.37
        longitude                 (location) float64 40B 11.19 11.29 11.3 11.39 11.5
        altitude                  (location) float64 40B 2.372e+03 ... 2.066e+03
        slope_inclination         (slope) float64 40B 0.0 38.0 38.0 38.0 38.0
        slope_a

## 1. Overview of dimensions
Before filtering, check which dimensions and coordinates the dataset contains:

In [72]:
print(ds.sizes)
print(ds.coords)

Frozen({'location': 5, 'time': 416, 'slope': 5, 'realization': 1, 'layer': 33})
Coordinates:
  * location           (location) int64 40B 0 1 2 3 4
  * time               (time) datetime64[ns] 3kB 2024-01-16T05:00:00 ... 2024...
  * realization        (realization) int64 8B 0
  * layer              (layer) int64 264B 0 1 2 3 4 5 6 ... 26 27 28 29 30 31 32
  * slope              (slope) int64 40B 0 1 2 3 4
    latitude           (location) float64 40B 47.15 47.44 47.15 47.44 47.37
    longitude          (location) float64 40B 11.19 11.29 11.3 11.39 11.5
    altitude           (location) float64 40B 2.372e+03 1.749e+03 ... 2.066e+03
    slope_inclination  (slope) float64 40B 0.0 38.0 38.0 38.0 38.0
    slope_azimuth      (slope) float64 40B 0.0 180.0 270.0 0.0 90.0
    z                  (location, time, slope, realization, layer) float32 1MB ...


Typical **dimensions**:

- `location` – one or several sites or grid points
- `time` – datetime stamps
- `slope` – inclination and azimuth combinations
- `realization` – simulation variant
- `layer` – vertical snow layers

**Coordinates** are one-dimensional arrays that label the positions along each dataset dimension. They define the values you use to index or align data — for example, the actual timestamps along time, the site names and their geographical coordinates along location, or slope angles along slope. In xarray, coordinates make datasets self-describing, so selections and merges can be done by meaningful labels instead of just integer indices.

## 2. Subsetting by coordinates

`xarray` provides two main selectors:

- `.isel()` – index-based (integer positions)
- `.sel()` – label-based (coordinate values)

### 2.1 Selecting by index position

In [47]:
# Select the last time step third location and first slope
subset = ds.isel(time=-1, location=2, slope=0)
print(subset['density'].isel(layer=slice(0,3)))

<xarray.DataArray 'density' (realization: 1, layer: 3)> Size: 12B
array([[246.1, 251. , 210.1]], dtype=float32)
Coordinates:
    location           int64 8B 2
    time               datetime64[ns] 8B 2024-02-02T12:00:00
  * realization        (realization) int64 8B 0
  * layer              (layer) int64 24B 0 1 2
    slope              int64 8B 0
    latitude           float64 8B 47.15
    longitude          float64 8B 11.3
    altitude           float64 8B 1.86e+03
    slope_inclination  float64 8B 0.0
    slope_azimuth      float64 8B 0.0
    z                  (realization, layer) float32 12B -8.68 -6.85 -5.33
Attributes:
    standard_name:  mass_density_of_snow
    units:          kg m-3
    long_name:      mass density of snow


Note: When you select a single element along a dimension (for example, `subset.isel(realization=0))`, xarray automatically *squeezes* that dimension out of the dataset. The dimension disappears from the list of dimensions (e.g., from `(realization: 1, layer: 3)` to `(layer: 3)`), but its coordinate (e.g., `realization = 0`) is still preserved in the metadata.

If you want to keep the dimension (so it still shows as length 1), wrap the index in a list:
```
ds.isel(realization=[0])
```
This can be important when chaining selections — once a dimension is dropped, you can no longer subset along it:
```
subset = ds.isel(slope=0)
subset.isel(slope=0)  # this will fail, because 'slope' is no longer a dimension
```

### 2.2 Selecting by coordinate values

In [ ]:
# Select by timestamp (label-based)
subset = ds.sel(time="2024-02-02 10:00")

2024-02-02T10:00:00.000000000
Frozen({'location': 5, 'slope': 5, 'realization': 1, 'layer': 33})


Time selection is flexible---you don’t need to type the full timestamp exactly. For example, you can specify partial strings such as `"2024-02-02"` (to select all hours of that day) or `"2024-02-02T10:00"` (to select a specific hour). 

You can also select using other time representations than `str`, such as `datetime.datetime` or `pandas.Timestamp`:

In [96]:
import pandas as pd
subset = ds.sel(time=pd.Timestamp("2024-02-02 10:00"))

from datetime import datetime
subset = ds.sel(time=datetime(2024, 2, 2, 10))

Or you can also select time ranges using Python-style slices:

In [78]:
subset = ds.sel(time=slice('2024-01-22T10:00', '2024-01-25T10:00'))

### 2.2 Subsetting with boolean expressions
You can also subset a dataset using boolean expressions on coordinate values for flexible, condition-based filtering across one or more dimensions.

In [84]:
subset = ds.sel(slope=(ds.slope_azimuth == 180))

In [86]:
subset = ds.sel(location=(ds.altitude > 2000))

In [93]:
subset = ds.sel(location=(ds.latitude > 47.3) & (ds.longitude < 11.4), 
                time=(ds.time > pd.Timestamp('2024-01-25T10:00')),
                slope=(ds.slope_azimuth == 180))

## 3. Masking and filtering by data conditions

Use `.where()` to mask or filter by values within variables. For example, retain only layers with density above 200 kg/m³:

In [ ]:
# First create an illustrative data subset:
sub = ds.isel(location=0, time=-1, slope=0)

In [ ]:
# Then filter based on condition:
dense = sub['density'].where(sub['density'] > 200)
dense.values

array([[214.2, 209.3, 223.8, 228.1, 251.4,   nan, 224.4, 225.3, 200.6,
          nan,   nan,   nan,   nan,   nan,   nan,   nan,   nan,   nan,
          nan,   nan,   nan,   nan,   nan,   nan,   nan,   nan,   nan,
          nan,   nan,   nan,   nan,   nan,   nan]], dtype=float32)

Masking via `.where()` keeps original shape but fills non-matching elements with `NaN`. To drop them entirely, add `drop=True`.

In [106]:
dense_drop = sub['density'].where(sub['density'] > 200, drop=True)
dense_drop.values

array([[214.2, 209.3, 223.8, 228.1, 251.4, 224.4, 225.3, 200.6]],
      dtype=float32)

::::{dropdown} How many profiles with `HS > 0` exist in `ds`?
:open: false

```python
# filter
ds = ds.compute_hs()
ds_snow_on_ground = ds.where((ds['HS'].notnull()) & (ds['HS'] > 0), drop=True)

# calculate
import numpy as np
dimensions = [dim for dim in ds_snow_on_ground.dims if dim != 'layer']
np.prod([ds_snow_on_ground[dim].size for dim in dimensions])
```

::::

## 4. Classification extension

`xsnow` includes a [classification extension](../api/_generated/xsnow.extensions.classification) that builds on simple masking to support criteria-based selection and multi-class classification.

It lets you:

 * create boolean masks from readable expressions (e.g., "density > 300 & grain_size < 0.5"),
 * attach those masks as coordinates or data variables for tidy subsetting, and
 * derive categorical classes (e.g., stability bins) from numeric variables.

See the API reference for details and examples.

## 5. Summary

- Use `.isel()` and `.sel()` for standard selection and conditional subsetting 
- Use `.where()` for conditional filtering
- Inspect `ds.sizes` and `ds.coords` to understand your selection context
